# Day 12: Implement a semantic search script

Welcome to Day 12! Today, we transition from foundational LLM usage to a core pillar of modern AI Engineering: **Semantic Search**. This is the mechanism that powers Retrieval-Augmented Generation (RAG) and allows applications to "understand" the meaning of queries rather than just matching keywords.


## Core Theory (Just-in-Time)

### What is Semantic Search?
Traditional search engines (like Elasticsearch using BM25) rely on **keyword matching**. If you search for "dog", it looks for the exact word "dog". 

**Semantic search**, on the other hand, understands *intent and context*. If you search for "canine companion", a semantic search engine knows this is related to "dog" even if the words don't match.

### How does it work?
1. **Embeddings:** Text is converted into dense vector representations (arrays of floating-point numbers) using an embedding model. These vectors represent the semantic meaning of the text.
2. **Vector Space:** Similar concepts are placed close together in a high-dimensional vector space.
3. **Similarity Search:** When a user queries the system, the query is also converted into a vector. The system then calculates the distance (e.g., Cosine Similarity, Euclidean Distance) between the query vector and all document vectors in the database. The closest vectors are returned as the most relevant results.

### AI Security Implications
- **PII Leakage:** Embeddings can implicitly encode Personally Identifiable Information (PII) if the source documents contain them. Before vectorizing documents, run a PII redaction step.
- **Access Control:** A vector DB doesn't inherently know who is searching. If you mix public and private data in the same collection, a user might retrieve semantic matches they shouldn't see. Always implement metadata filtering (e.g., `user_id` or `tenant_id`) alongside semantic search to strictly isolate data.


## Code Implementation

We will implement semantic search using Qdrant (a fast, Rust-based vector database) and LangChain's HuggingFace integrations. We progress from a basic standalone script to a robust class-based utility.


### Basic Implementation
A minimal script to demonstrate the raw mechanics of creating embeddings and finding the closest match using an in-memory Qdrant instance.


In [1]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
import uuid

# 1. Initialize Embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Initialize in-memory Qdrant client & collection
client = QdrantClient(":memory:")
client.create_collection(
    collection_name="basic_collection",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

# 3. Create Vector Store
vector_store = QdrantVectorStore(
    client=client,
    collection_name="basic_collection",
    embedding=embeddings,
)

# 4. Ingest Documents
docs = [
    Document(page_content="The quick brown fox jumps over the lazy dog."),
    Document(page_content="Artificial Intelligence is transforming software.")
]
uuids = [str(uuid.uuid4()) for _ in range(len(docs))]
vector_store.add_documents(documents=docs, ids=uuids)

# 5. Perform Search
results = vector_store.similarity_search("Machine learning tools", k=1)
print("Basic Search Result:", results[0].page_content)


/app/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1796.08it/s]

Basic Search Result: Artificial Intelligence is transforming software.


### Medium Implementation
Introducing OOP principles to encapsulate state (the client and embeddings) and defining distinct methods for ingestion and search.


In [2]:
from typing import List
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
import uuid

class SimpleSemanticSearch:
    def __init__(self, collection_name: str):
        self.collection_name = collection_name
        self.embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
        self.client = QdrantClient(":memory:")
        
        self.client.create_collection(
            collection_name=self.collection_name,
            vectors_config=VectorParams(size=384, distance=Distance.COSINE),
        )
        
        self.vector_store = QdrantVectorStore(
            client=self.client,
            collection_name=self.collection_name,
            embedding=self.embeddings,
        )
        
    def ingest(self, texts: List[str]) -> None:
        docs = [Document(page_content=text) for text in texts]
        ids = [str(uuid.uuid4()) for _ in range(len(docs))]
        self.vector_store.add_documents(documents=docs, ids=ids)
        
    def search(self, query: str, k: int = 1) -> List[Document]:
        return self.vector_store.similarity_search(query, k=k)

searcher = SimpleSemanticSearch("medium_collection")
searcher.ingest(["Apples are fruits.", "Python is a language."])
res = searcher.search("coding", k=1)
print("Medium Search Result:", res[0].page_content)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1685.91it/s]

Medium Search Result: Python is a language.


### Advanced Implementation
Production-ready implementation featuring type hinting, docstrings, structured metadata filtering (for AI Security multi-tenancy), and a fallback mechanism to handle empty results.


In [3]:
from typing import List, Dict, Any, Tuple, Optional
import logging
import uuid

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, Filter, FieldCondition, MatchValue

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class SecureSemanticSearcher:
    """
    A production-ready semantic searcher with metadata filtering capabilities 
    to ensure strict tenant isolation (AI Security pattern).
    """
    
    def __init__(self, collection_name: str = "secure_docs", embedding_dim: int = 384):
        """
        Initialize the secure search utility.
        
        Args:
            collection_name (str): Name of the Qdrant collection.
            embedding_dim (int): Dimensionality of the embedding vectors.
        """
        self.collection_name = collection_name
        
        try:
            self.embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
            self.client = QdrantClient(":memory:")
            
            self.client.create_collection(
                collection_name=self.collection_name,
                vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE),
            )
            
            self.vector_store = QdrantVectorStore(
                client=self.client,
                collection_name=self.collection_name,
                embedding=self.embeddings,
            )
            logger.info(f"Initialized secure searcher with collection '{self.collection_name}'")
        except Exception as e:
            logger.error(f"Failed to initialize vector store: {e}")
            raise

    def ingest_documents(self, documents: List[Dict[str, Any]]) -> None:
        """
        Ingests documents with mandatory metadata (e.g., tenant_id).
        
        Args:
            documents (List[Dict[str, Any]]): List of dicts containing 'text' and 'metadata'.
                                              'metadata' MUST contain a 'tenant_id'.
        """
        docs = []
        for item in documents:
            meta = item.get("metadata", {})
            if "tenant_id" not in meta:
                raise ValueError("Security Violation: 'tenant_id' is mandatory in metadata.")
                
            docs.append(
                Document(
                    page_content=item["text"],
                    metadata=meta
                )
            )
            
        ids = [str(uuid.uuid4()) for _ in range(len(docs))]
        self.vector_store.add_documents(documents=docs, ids=ids)
        logger.info(f"Successfully ingested {len(docs)} documents.")

    def search_for_tenant(
        self, 
        query: str, 
        tenant_id: str, 
        top_k: int = 3
    ) -> List[Tuple[Document, float]]:
        """
        Executes a semantic search strictly isolated to the specified tenant_id.
        Includes a graceful fallback if no results are found.
        
        Args:
            query (str): The search query.
            tenant_id (str): The identifier for the tenant (security barrier).
            top_k (int): The maximum number of results to return.
            
        Returns:
            List[Tuple[Document, float]]: Ranked matching documents and their similarity scores.
        """
        # Construct Qdrant filter for hard isolation
        tenant_filter = Filter(
            must=[
                FieldCondition(
                    key="metadata.tenant_id",
                    match=MatchValue(value=tenant_id),
                )
            ]
        )
        
        try:
            results = self.vector_store.similarity_search_with_score(
                query, 
                k=top_k, 
                filter=tenant_filter
            )
            
            # Fallback mechanism if no results match the filter
            if not results:
                logger.warning(f"No results found for query '{query}' under tenant '{tenant_id}'.")
                return [(Document(page_content="No relevant context found in your documents.", metadata={"tenant_id": tenant_id}), 0.0)]
                
            return results
        except Exception as e:
            logger.error(f"Search execution failed: {e}")
            return []

# Execution block
if __name__ == "__main__":
    data = [
        {"text": "Acme Corp's Q3 revenue was $5M.", "metadata": {"tenant_id": "acme_01", "doc_type": "financial"}},
        {"text": "Globex's new AI strategy focuses on RAG.", "metadata": {"tenant_id": "globex_02", "doc_type": "strategy"}},
        {"text": "Acme Corp plans to acquire a new startup.", "metadata": {"tenant_id": "acme_01", "doc_type": "strategy"}}
    ]
    
    secure_db = SecureSemanticSearcher()
    secure_db.ingest_documents(data)
    
    print("\n--- Globex User searching for Acme Data (Should Fail/Fallback) ---")
    unauthorized_search = secure_db.search_for_tenant("What is Acme's revenue?", tenant_id="globex_02")
    print(unauthorized_search[0][0].page_content)
    
    print("\n--- Acme User searching for their own strategy ---")
    authorized_search = secure_db.search_for_tenant("What are our strategic plans?", tenant_id="acme_01")
    print(f"Score: {authorized_search[0][1]:.4f} | Content: {authorized_search[0][0].page_content}")


INFO:sentence_transformers.base.model:No device provided, using cpu


INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"


...

INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/2_Normalize/config.json "HTTP/1.1 404 Not Found"


INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2 "HTTP/1.1 200 OK"


INFO:__main__:Initialized secure searcher with collection 'secure_docs'


INFO:__main__:Successfully ingested 3 documents.



--- Globex User searching for Acme Data (Should Fail/Fallback) ---
Globex's new AI strategy focuses on RAG.

--- Acme User searching for their own strategy ---
Score: 0.2890 | Content: Acme Corp plans to acquire a new startup.


## Common Pitfalls (Production Considerations)

1. **Embedding Dimension Mismatches:** You must ensure the vector size defined in your Qdrant collection exactly matches the output dimension of your chosen embedding model. If you switch models (e.g., from `all-MiniLM-L6-v2` [384] to OpenAI `text-embedding-3-small` [1536]), you must recreate the collection or use different namespaces.
2. **Chunking Strategy Ignored:** In this example, our documents were already single sentences. In production, feeding massive documents directly into an embedding model will truncate them (due to token limits) and dilute the semantic meaning. You *must* split large text into smaller chunks before vectorizing.
3. **Missing Access Controls:** Never query a global vector DB without metadata filtering if the data belongs to different users. Always inject the user's `tenant_id` as a hard filter in the search query to prevent data leakage.


## Practical Lab / Homework

**Instructions:**
1. Create a class `ProductSearchEngine`.
2. Initialize it with an in-memory Qdrant client and `HuggingFaceEmbeddings`.
3. Write a method to ingest a list of dictionaries representing products. Each dictionary should have `name`, `description`, and `category`.
   - Store the `description` as the main `page_content`.
   - Store `name` and `category` as metadata.
4. Implement a `search_by_category` method that accepts a user query and a category string, returning the closest matching product but *only* if it falls within the specified category using Qdrant Filters.
5. Use proper OOP principles, explicit type hinting, and add a brief docstring.
6. Record a 2-minute async video walkthrough explaining how your Qdrant filter enforces the category constraint.


In [4]:
# Practical Lab Implementation
from typing import List, Dict, Any, Tuple
import uuid
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, Filter, FieldCondition, MatchValue

class ProductSearchEngine:
    """
    A product search engine that filters semantic results by category.
    """
    
    def __init__(self):
        self.embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
        self.client = QdrantClient(":memory:")
        self.collection_name = "products"
        
        self.client.create_collection(
            collection_name=self.collection_name,
            vectors_config=VectorParams(size=384, distance=Distance.COSINE),
        )
        
        self.vector_store = QdrantVectorStore(
            client=self.client,
            collection_name=self.collection_name,
            embedding=self.embeddings,
        )
        
    def ingest_products(self, products: List[Dict[str, str]]) -> None:
        """
        Ingests product descriptions and metadata.
        """
        docs = []
        for prod in products:
            doc = Document(
                page_content=prod["description"],
                metadata={"name": prod["name"], "category": prod["category"]}
            )
            docs.append(doc)
            
        ids = [str(uuid.uuid4()) for _ in range(len(docs))]
        self.vector_store.add_documents(documents=docs, ids=ids)
        
    def search_by_category(self, query: str, category: str, top_k: int = 1) -> List[Tuple[Document, float]]:
        """
        Searches for a product using semantic similarity, strictly filtered by category.
        """
        category_filter = Filter(
            must=[
                FieldCondition(
                    key="metadata.category",
                    match=MatchValue(value=category)
                )
            ]
        )
        
        return self.vector_store.similarity_search_with_score(
            query, 
            k=top_k,
            filter=category_filter
        )

if __name__ == "__main__":
    inventory = [
        {"name": "SuperGamer PC", "description": "High-end desktop for competitive gaming.", "category": "Electronics"},
        {"name": "ErgoChair Pro", "description": "Ergonomic office chair with lumbar support.", "category": "Furniture"},
        {"name": "DevLaptop", "description": "Lightweight machine perfect for writing code.", "category": "Electronics"}
    ]
    
    engine = ProductSearchEngine()
    engine.ingest_products(inventory)
    
    print("\n--- Searching for 'coding computer' in Electronics ---")
    results = engine.search_by_category("coding computer", category="Electronics")
    if results:
        print(f"Found: {results[0][0].metadata['name']} - {results[0][0].page_content}")


INFO:sentence_transformers.base.model:No device provided, using cpu


INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"


...

INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/2_Normalize/config.json "HTTP/1.1 404 Not Found"


INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2 "HTTP/1.1 200 OK"



--- Searching for 'coding computer' in Electronics ---
Found: DevLaptop - Lightweight machine perfect for writing code.


## Reference Links
- [LangChain Qdrant Integration](https://python.langchain.com/docs/integrations/vectorstores/qdrant)
- [Qdrant Documentation: Filtering](https://qdrant.tech/documentation/concepts/filtering/)
- [HuggingFace Embeddings in LangChain](https://python.langchain.com/docs/integrations/text_embedding/huggingfacehub)
